# MTL transition HFSS wave-port S-parameters

This notebook builds one transition coupon directly from public transition cells and
assigns physical wave ports on external faces only. `RUN_PREPARE` and
`RUN_SOLVER` are `False` by default for safe GUI review.

## Setup and Imports

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as package_version
from math import cos, pi, sin
from pathlib import Path
from time import perf_counter, sleep
from typing import Any

import gdsfactory as gf
import pandas as pd
import skrf
from ansys.aedt.core import Hfss
from IPython.display import HTML, clear_output, display
from matplotlib.path import Path as PolygonPath

import orpen_sc_pdk
from orpen_sc_pdk.cells.airbridge import airbridge
from orpen_sc_pdk.cells.cpw import mtl_bend_bend_transition, mtl_straight_bend_transition
from orpen_sc_pdk.simulation.aedt import aedt_material_name_from_physical_key
from orpen_sc_pdk.tech import (
    AIRBRIDGE_THICKNESS_UM,
    AIRBRIDGE_VIA_THICKNESS_UM,
    CPW_DRAW,
    CPW_GROUND_MASK,
    OUTER_VACUUM_THICKNESS_UM,
    SUBSTRATE_THICKNESS_UM,
)

REPO_ROOT = Path(orpen_sc_pdk.__file__).resolve().parent.parent
if not (REPO_ROOT / "orpen_sc_pdk").is_dir():
    raise RuntimeError("Active orpen_sc_pdk checkout is invalid for notebook replay.")
orpen_sc_pdk.activate()

## Setup and Run Controls

In [ ]:
TRANSITION_KIND = "straight_bend"  # {"straight_bend", "bend_bend"}
MTL_LEAD_LENGTH_UM = 400.0
SINGLE_TRACE_LEAD_LENGTH_UM = 600.0
MTL_DEEMBED_LENGTH_UM = 400.0
SINGLE_TRACE_DEEMBED_LENGTH_UM = 600.0
FINITE_GROUND_WIDTH_UM = 80.0
SUBSTRATE_KEY = "Si"

# Diagnostic only: tie the three finite-ground rails together halfway along the
# uniform coupled lead. The 3 um center landing matches the inter-trace ground.
ADD_GROUND_AIRBRIDGE = False
AIRBRIDGE_WIDTH_UM = 12.0
AIRBRIDGE_VIA_SIZE_UM = 3.0

# Experimental seam-center terminalization can be switched on/off here.
USE_SEAM_CENTER_GROUND_TERMINAL = True

FREQUENCY_START_GHZ = 3.0
FREQUENCY_STOP_GHZ = 8.0
FREQUENCY_POINT_COUNT = 20_000
SWEEP_TYPE = "Fast"
SWEEP_NAME = "S"

MAX_ADAPTIVE_PASSES = 99
MINIMUM_CONVERGED_PASSES = 2
MAX_DELTA_S = 0.02
MAXIMUM_DELTA_ZO_PERCENT = 1.0
CPW_CONDUCTOR_LENGTH_MESH_UM = 7.0
CPW_CONDUCTOR_LENGTH_MESH_MAX_ADDITIONAL_ELEMENTS = 1_000_000
GROUND_SURFACE_MESH_LEVEL = 9

SOLVER_SETUP_NAME = "Setup1"
SOLUTION_TYPE = "Terminal"
AEDT_VERSION = "2024.2"
NON_GRAPHICAL = True
CLOSE_DESKTOP = False
PYAEDT_EXPECTED_VERSION = "1.3.0"
PYAEDT_DOCS_URL = "https://aedt.docs.pyansys.com/version/1.3.0/"
PYAEDT_SOURCE_URL = "https://github.com/ansys/pyaedt/tags"


def _resolve_pyaedt_version(expected_version: str) -> str:
    try:
        installed_version = package_version("pyaedt")
    except PackageNotFoundError as exc:
        raise RuntimeError(
            "PyAEDT distribution 'pyaedt' is not installed in this environment; "
            f"expected version {expected_version!r}."
        ) from exc

    if installed_version != expected_version:
        message = (
            f"PyAEDT version mismatch: installed {installed_version!r}, "
            f"expected {expected_version!r}. "
            f"See {PYAEDT_DOCS_URL} and {PYAEDT_SOURCE_URL}."
        )
        raise RuntimeError(message)

    return installed_version


PYAEDT_INSTALLED_VERSION = _resolve_pyaedt_version(PYAEDT_EXPECTED_VERSION)

# Execution controls are intentionally off in committed notebook.
RUN_PREPARE = False
RUN_SOLVER = False
RUN_AEDT = RUN_PREPARE or RUN_SOLVER

CASE_NAME = (
    f"{TRANSITION_KIND}_mtl{MTL_LEAD_LENGTH_UM:g}_single{SINGLE_TRACE_LEAD_LENGTH_UM:g}_"
    f"ground{FINITE_GROUND_WIDTH_UM:g}_mtlde{MTL_DEEMBED_LENGTH_UM:g}_"
    f"singlede{SINGLE_TRACE_DEEMBED_LENGTH_UM:g}_dzo{MAXIMUM_DELTA_ZO_PERCENT:g}pct_terminalmodes"
    f"{'_airbridge' if ADD_GROUND_AIRBRIDGE else ''}"
    f"{'_seam5p' if USE_SEAM_CENTER_GROUND_TERMINAL else ''}"
)
RUN_DIR = (
    REPO_ROOT
    / "build"
    / "simulation"
    / "aedt"
    / "mtl_transition_hfss_sparameters"
    / TRANSITION_KIND
    / CASE_NAME
)
GDS_PATH = RUN_DIR / f"{CASE_NAME}.gds"
PROJECT_PATH = RUN_DIR / f"{CASE_NAME}.aedt"
TOUCHSTONE_PATH = RUN_DIR / (
    "mtl_transition.s5p" if USE_SEAM_CENTER_GROUND_TERMINAL else "mtl_transition.s4p"
)
PORT_ZO_PATH = RUN_DIR / "modal_port_zo_zpi.csv"
TIMING_PATH = RUN_DIR / "solve_timing.json"
S_PARAMETER_PLOT_PATH = RUN_DIR / "sparameters.html"
METADATA_PATH = RUN_DIR / "mtl_transition_metadata.json"
ACF_PATH = REPO_ROOT / "notebooks" / "AEDTSimulation" / "HFSS_Local.acf"

# HFSS-only GDS layers used only for this coupon.
HFSS_SIGNAL_P_LAYER = (905, 0)
HFSS_SIGNAL_R_LAYER = (906, 0)
HFSS_GROUND_LAYER = (907, 0)
HFSS_SUBSTRATE_LAYER = (908, 0)
HFSS_AIRBRIDGE_LAYER = (909, 0)
HFSS_AIRBRIDGE_VIA_LAYER = (910, 0)

# Region padding order is [+X, -X, +Y, -Y, +Z, -Z].
REGION_PAD = OUTER_VACUUM_THICKNESS_UM

# Terminal names as emitted by PyAEDT's created boundary objects.
CANONICAL_TERMINAL_ORDER = (
    (
        "o1",
        "o2",
        "g_center",
        "o3",
        "o4",
    )
    if USE_SEAM_CENTER_GROUND_TERMINAL
    else ("o1", "o2", "o3", "o4")
)
MIN_STRAIGHT_BEND_LEAD_UM = FINITE_GROUND_WIDTH_UM + 6.0 + 7.0 / 2


class BuildError(RuntimeError):
    """Raised when geometry, port mapping, or boundary creation is not exact."""


@dataclass(frozen=True)
class PortRecord:
    physical_port: str
    object_name: str
    side: str
    axis: str
    side_sign: int
    port_center_um: tuple[float, float, float]


@dataclass(frozen=True)
class TerminalRecord:
    physical_port: str
    boundary_name: str
    terminal_names: list[str]
    terminal_objects: dict[str, str]
    reference_conductors: list[str]
    face_id: int
    expected_terminal_count: int
    mode_count: int


def _normalize_transition_kind(kind: str) -> str:
    normalized = str(kind).strip().lower().replace("-", "_").replace(" ", "_")
    if normalized not in {"straight_bend", "bend_bend"}:
        raise ValueError(f"Unsupported TRANSITION_KIND: {kind!r}.")
    return normalized


def _deembed_mm(deembed_length_um: float) -> float:
    if deembed_length_um < 0:
        raise ValueError(f"deembed_length_um must be >= 0, got {deembed_length_um!r}.")
    return deembed_length_um / 1000.0


def _deembed_um(deembed_length_um: float) -> str:
    if deembed_length_um < 0:
        raise ValueError(f"deembed_length_um must be >= 0, got {deembed_length_um!r}.")
    return f"{deembed_length_um:g}um"


def _normalization_index(
    source_port_names: list[str], canonical_order: tuple[str, ...]
) -> tuple[list[int], list[str], list[str]]:
    if len(source_port_names) != len(canonical_order):
        raise BuildError(
            f"Touchstone port count mismatch: expected {len(canonical_order)} ports, "
            f"got {len(source_port_names)}."
        )

    if len(set(source_port_names)) != len(source_port_names):
        raise BuildError(f"Touchstone export contains duplicate port names: {source_port_names!r}.")

    canonical = list(canonical_order)
    missing = [name for name in canonical if name not in source_port_names]
    extra = [name for name in source_port_names if name not in canonical]
    if missing or extra:
        raise BuildError(
            "Touchstone port names are not a permutation of canonical order. "
            f"expected={canonical!r}, got={source_port_names!r}, "
            f"missing={missing!r}, extra={extra!r}."
        )

    index = [source_port_names.index(name) for name in canonical]
    return index, source_port_names, canonical


def _normalize_touchstone_ports(
    touchstone_path: Path,
    canonical_port_order: tuple[str, ...],
    *,
    form: str = "ma",
    r_ref: float = 50.0,
) -> dict[str, list[str] | bool]:
    network = skrf.Network(str(touchstone_path))
    if not network.port_names:
        raise BuildError(f"Touchstone export at {touchstone_path!s} has no readable port names.")

    index, original_order, normalized_order = _normalization_index(
        list(network.port_names),
        canonical_port_order,
    )

    if list(network.port_names) != normalized_order:
        network.s = network.s[:, index, :][:, :, index]
        network.z0 = network.z0[:, index]
        network.port_names = normalized_order
        network.write_touchstone(
            str(touchstone_path),
            form=form,
            r_ref=r_ref,
            write_z0=False,
            skrf_comment=True,
        )
    return {
        "original_port_order": original_order,
        "normalized_port_order": normalized_order,
        "was_reordered": original_order != normalized_order,
    }


def _side_map(kind: str) -> dict[str, str]:
    if kind == "straight_bend":
        # seam at -X, outer1 at +Y, outer2 at +X
        return {
            "o1": "-X",
            "o2": "-X",
            "o3": "+Y",
            "o4": "+X",
        }
    if kind == "bend_bend":
        # seam at -X, outer1 at +Y, outer2 at -Y
        return {
            "o1": "-X",
            "o2": "-X",
            "o3": "+Y",
            "o4": "-Y",
        }
    raise BuildError(f"Unknown transition kind {kind!r}.")


def _object_map() -> dict[str, str]:
    return {
        "o1": "signal_1",
        "o2": "signal_2",
        "o3": "signal_2",
        "o4": "signal_1",
    }


def _build_transition_component(
    kind: str,
    mtl_lead_length_um: float,
    single_trace_lead_length_um: float,
) -> tuple[gf.Component, list[str]]:
    if kind == "straight_bend":
        if single_trace_lead_length_um < MIN_STRAIGHT_BEND_LEAD_UM:
            raise BuildError(
                "straight_bend single-trace lead is too short for the requested "
                "boundary-face ground: "
                f"need at least {MIN_STRAIGHT_BEND_LEAD_UM:g} um for W7/S6 and "
                f"{FINITE_GROUND_WIDTH_UM:g} um ground, got {single_trace_lead_length_um:g} um."
            )
        component = mtl_straight_bend_transition(
            straight_length=100.0,
            inter_trace_ground_width=3.0,
            bend_radius=100.0,
            mtl_lead_length=mtl_lead_length_um,
            single_trace_lead_length=single_trace_lead_length_um,
        )
    elif kind == "bend_bend":
        component = mtl_bend_bend_transition(
            bend_radius=100.0,
            inter_trace_ground_width=3.0,
            mtl_lead_length=mtl_lead_length_um,
            single_trace_lead_length=single_trace_lead_length_um,
        )
    else:
        raise ValueError(f"Unsupported transition kind {kind!r}.")

    expected_ports = {"o1", "o2", "o3", "o4"}
    observed_ports = {port.name for port in component.ports}
    if observed_ports != expected_ports:
        raise BuildError(
            "Transition component ports mismatch. "
            f"Expected {sorted(expected_ports)}, got {sorted(observed_ports)}."
        )
    ordered = list(component.info.get("ordered_port_names", ()))
    if ordered != ["o1", "o2", "o3", "o4"]:
        raise BuildError(
            "Transition component port ordering changed from accepted contract: "
            f"expected ['o1','o2','o3','o4'], got {ordered!r}."
        )
    return component, ordered


def _signal_polygon_at_port(signal_polygons: list[Any], *, port: gf.Port) -> Any:
    if port.orientation is None:
        raise BuildError(f"Port {port.name!r} must expose orientation.")
    probe_distance_um = float(port.width) / 4
    angle = float(port.orientation) * pi / 180
    probe = (
        float(port.x) - probe_distance_um * cos(angle),
        float(port.y) - probe_distance_um * sin(angle),
    )
    matches = [polygon for polygon in signal_polygons if PolygonPath(polygon).contains_point(probe)]
    if len(matches) != 1:
        raise BuildError(
            f"Port {port.name!r} must match exactly one signal polygon, got {len(matches)}."
        )
    return matches[0]


def _select_region_layer_ports(
    component: gf.Component,
) -> tuple[list[Any], list[Any]]:
    xs = gf.get_cross_section("cpw_6_7_6")
    draw_layer = tuple(int(value) for value in gf.get_layer_tuple(xs[CPW_DRAW].layer))
    gm_layer = tuple(int(value) for value in gf.get_layer_tuple(xs[CPW_GROUND_MASK].layer))

    polygons = component.dup()
    signal_polygons = polygons.get_polygons_points(merge=True, by="tuple", layers=[draw_layer])[
        draw_layer
    ]
    ground_mask_polygons = polygons.get_polygons_points(merge=True, by="tuple", layers=[gm_layer])[
        gm_layer
    ]
    if not signal_polygons:
        raise BuildError("Transition has no signal region on CPW draw layer.")
    if not ground_mask_polygons:
        raise BuildError("Transition has no ground-mask region on CPW ground mask layer.")
    return signal_polygons, ground_mask_polygons


def _build_hfss_coupon_from_transition(
    component: gf.Component,
    *,
    transition_kind: str,
    mtl_lead_length_um: float,
    single_trace_lead_length_um: float,
    mtl_deembed_length_um: float,
    single_trace_deembed_length_um: float,
) -> tuple[gf.Component, list[PortRecord]]:
    signal_polygons, ground_mask_polygons = _select_region_layer_ports(component)

    selected_signals = {
        "signal_1": _signal_polygon_at_port(signal_polygons, port=component.ports["o1"]),
        "signal_2": _signal_polygon_at_port(signal_polygons, port=component.ports["o2"]),
    }
    if selected_signals["signal_1"] is selected_signals["signal_2"]:
        raise BuildError("Coupled traces collapsed; expected separate seam traces.")

    ground_mask = gf.Component()
    for polygon in ground_mask_polygons:
        ground_mask.add_polygon(polygon, layer=HFSS_GROUND_LAYER)

    # Lead length controls the longitudinal uniform CPW.  Finite-ground width is
    # transverse: extend the coupon on sides without a Wave Port, while keeping
    # every port side flush with its reference plane.
    mask_bbox = ground_mask.dbbox()
    port_sides = _side_map(transition_kind)
    side_ports = {
        side: [
            component.ports[name] for name, mapped_side in port_sides.items() if mapped_side == side
        ]
        for side in {"-X", "+X", "-Y", "+Y"}
    }
    left = (
        min(float(port.x) for port in side_ports["-X"])
        if side_ports["-X"]
        else float(mask_bbox.left) - FINITE_GROUND_WIDTH_UM
    )
    right = (
        max(float(port.x) for port in side_ports["+X"])
        if side_ports["+X"]
        else float(mask_bbox.right) + FINITE_GROUND_WIDTH_UM
    )
    bottom = (
        min(float(port.y) for port in side_ports["-Y"])
        if side_ports["-Y"]
        else float(mask_bbox.bottom) - FINITE_GROUND_WIDTH_UM
    )
    top = (
        max(float(port.y) for port in side_ports["+Y"])
        if side_ports["+Y"]
        else float(mask_bbox.top) + FINITE_GROUND_WIDTH_UM
    )
    size = (right - left, top - bottom)

    coupon_rectangle = gf.components.rectangle(size=size, layer=HFSS_SUBSTRATE_LAYER)
    coupon_rectangle_ref = gf.Component()
    coupon_rectangle_ref.add_ref(coupon_rectangle).dmove((left, bottom))
    finite_ground = gf.boolean(
        coupon_rectangle_ref,
        ground_mask,
        operation="not",
        layer=HFSS_GROUND_LAYER,
        layer1=HFSS_SUBSTRATE_LAYER,
        layer2=HFSS_GROUND_LAYER,
    )

    hfss_coupon = gf.Component()
    hfss_coupon.add_polygon(selected_signals["signal_1"], layer=HFSS_SIGNAL_P_LAYER)
    hfss_coupon.add_polygon(selected_signals["signal_2"], layer=HFSS_SIGNAL_R_LAYER)
    hfss_coupon.add_ref(finite_ground)
    substrate = hfss_coupon.add_ref(coupon_rectangle)
    substrate.dmove((left, bottom))

    airbridge_metadata: dict[str, Any] = {"enabled": False}
    if ADD_GROUND_AIRBRIDGE:
        seam_x = float(component.ports["o1"].x)
        seam_y = (float(component.ports["o1"].y) + float(component.ports["o2"].y)) / 2
        trace_offset = abs(float(component.ports["o2"].y) - float(component.ports["o1"].y)) / 2
        bridge_span = 2 * trace_offset
        bridge_x = seam_x + mtl_lead_length_um / 2

        for bridge_y in (seam_y - trace_offset, seam_y + trace_offset):
            bridge = hfss_coupon.add_ref(
                airbridge(
                    bridge_span=bridge_span,
                    bridge_width=AIRBRIDGE_WIDTH_UM,
                    via_size=AIRBRIDGE_VIA_SIZE_UM,
                    airbridge_draw_layer=HFSS_AIRBRIDGE_LAYER,
                    airbridge_via_layer=HFSS_AIRBRIDGE_VIA_LAYER,
                )
            )
            bridge.dmove((bridge_x, bridge_y))

        airbridge_metadata = {
            "enabled": True,
            "x_um": bridge_x,
            "bridge_span_um": bridge_span,
            "bridge_width_um": AIRBRIDGE_WIDTH_UM,
            "via_size_um": AIRBRIDGE_VIA_SIZE_UM,
            "role": "connect lower, center, and upper finite-ground rails",
        }

    hfss_coupon.add_port(
        name="o1",
        port=component.ports["o1"],
        layer=HFSS_SIGNAL_P_LAYER,
    )
    hfss_coupon.add_port(
        name="o2",
        port=component.ports["o2"],
        layer=HFSS_SIGNAL_R_LAYER,
    )
    hfss_coupon.add_port(
        name="o3",
        port=component.ports["o3"],
        layer=HFSS_SIGNAL_P_LAYER,
    )
    hfss_coupon.add_port(
        name="o4",
        port=component.ports["o4"],
        layer=HFSS_SIGNAL_R_LAYER,
    )
    hfss_coupon.flatten(merge=True)

    side_map = _side_map(transition_kind)
    hfss_coupon.info["hfss_coupon"] = {
        "transition_kind": transition_kind,
        "mtl_lead_length_um": float(mtl_lead_length_um),
        "single_trace_lead_length_um": float(single_trace_lead_length_um),
        "deembed_mtl_length_um": float(mtl_deembed_length_um),
        "deembed_single_trace_length_um": float(single_trace_deembed_length_um),
        "finite_ground_width_um": float(FINITE_GROUND_WIDTH_UM),
        "hfss_layers": {
            "signal_1": HFSS_SIGNAL_P_LAYER,
            "signal_2": HFSS_SIGNAL_R_LAYER,
            "finite_ground": HFSS_GROUND_LAYER,
            "substrate": HFSS_SUBSTRATE_LAYER,
            "airbridge": HFSS_AIRBRIDGE_LAYER,
            "airbridge_via": HFSS_AIRBRIDGE_VIA_LAYER,
        },
        "materials": {
            "signal_1": "PEC",
            "signal_2": "PEC",
            "finite_ground": "PEC",
            "substrate": "Si",
            "airbridge": "PEC",
            "airbridge_via": "PEC",
        },
        "airbridge": airbridge_metadata,
        "coupon_bbox_um": {
            "xmin": left,
            "xmax": right,
            "ymin": bottom,
            "ymax": top,
        },
    }

    port_records = [
        PortRecord(
            physical_port=name,
            object_name=_object_map()[name],
            side=side,
            axis=side[1],
            side_sign=-1 if side[0] == "-" else 1,
            port_center_um=tuple(float(v) for v in component.ports[name].center),
        )
        for name, side in side_map.items()
    ]
    return hfss_coupon, port_records


def _region_padding_for_kind(kind: str) -> list[float]:
    # PyAEDT assigns the Wave Port to the whole selected Region face, so XY padding would grow
    # the face aperture. Keep XY pads zero so both kinds share equal external cross-sections;
    # keep only Z padding explicit for vertical clearance.
    if kind not in {"straight_bend", "bend_bend"}:
        raise BuildError(f"Unsupported transition kind {kind!r}.")
    return [0.0, 0.0, 0.0, 0.0, REGION_PAD, REGION_PAD]


def _get_object_bbox(app: Hfss, object_name: str) -> dict[str, float]:
    obj = app.modeler.get_object_from_name(object_name)
    if not obj:
        raise BuildError(f"Missing AEDT object {object_name!r}.")
    bbox = obj.bounding_box
    if len(bbox) != 6:
        raise BuildError(f"Unexpected bounding_box result for {object_name!r}: {bbox!r}")
    return {
        "xmin": float(bbox[0]),
        "ymin": float(bbox[1]),
        "zmin": float(bbox[2]),
        "xmax": float(bbox[3]),
        "ymax": float(bbox[4]),
        "zmax": float(bbox[5]),
        "x_center": 0.5 * (float(bbox[0]) + float(bbox[3])),
        "y_center": 0.5 * (float(bbox[1]) + float(bbox[4])),
    }


def _select_central_seam_ground(
    app: Hfss,
    ground_references: list[str],
    seam_port_records: list[PortRecord],
) -> str:
    seam_ports = [record for record in seam_port_records if record.physical_port in {"o1", "o2"}]
    if len(seam_ports) != 2:
        raise BuildError(f"Expected seam ports o1/o2 in assignments, got {seam_ports!r}.")

    seam_candidates: list[tuple[str, float, float]] = []

    seam_axis = seam_ports[0].axis
    seam_transverse_idx = 1 if seam_axis == "X" else 0
    seam_position = seam_ports[0].port_center_um[0 if seam_axis == "X" else 1]
    seam_mid_transverse = sum(port.port_center_um[seam_transverse_idx] for port in seam_ports) / 2.0

    for ground_name in ground_references:
        obj = app.modeler.get_object_from_name(ground_name)
        if not obj:
            raise BuildError(f"Missing imported finite-ground object {ground_name!r}.")

        # The center rail is the only finite ground crossing the midpoint between
        # o1 and o2 on the seam plane; object numbering is not geometrical authority.
        seam_axis_points = [
            tuple(float(value) for value in vertex.position)
            for vertex in obj.vertices
            if abs(float(vertex.position[0 if seam_axis == "X" else 1]) - seam_position) <= 1e-6
        ]

        if not seam_axis_points:
            continue

        seam_transverse = [point[seam_transverse_idx] for point in seam_axis_points]
        seam_min = min(seam_transverse)
        seam_max = max(seam_transverse)
        if seam_min <= seam_mid_transverse <= seam_max:
            seam_candidates.append((ground_name, seam_min, seam_max))

    if len(seam_candidates) != 1:
        raise BuildError(
            "Expected exactly one central finite-ground seam candidate; "
            f"found {len(seam_candidates)}: {seam_candidates!r}, "
            f"from ground references {ground_references!r} at seam {seam_axis}{seam_position:g}."
        )

    return seam_candidates[0][0]


def _find_region_face_on_side(app: Hfss, side: str) -> int:
    obj = app.modeler.get_object_from_name("Region")
    faces = list(obj.faces)
    if not faces:
        raise BuildError("No faces found on imported 'Region'.")
    axis = 0 if side[1] == "X" else 1
    sign = -1 if side[0] == "-" else 1

    def score(face: Any) -> tuple[float, float]:
        center = getattr(face, "center", None)
        if not center:
            raise BuildError("Face on 'Region' has no center metadata.")
        axis_value = float(center[axis])
        return sign * axis_value, float(center[2])

    chosen = max(faces, key=score)
    return int(chosen.id)


def _create_wave_port(
    app: Hfss,
    *,
    ground_references: list[str],
    terminal_names: list[str],
    terminal_object_map: dict[str, str],
    physical_port: str,
    face_id: int,
    boundary_name: str,
    deembed_distance: float | str,
    prior_terminals: set[str],
) -> TerminalRecord:
    expected_terminal_count = len(terminal_names)
    boundary = app.wave_port(
        face_id,
        reference=ground_references,
        name=boundary_name,
        deembed=deembed_distance,
        terminals_rename=True,
    )
    if not boundary:
        raise BuildError(f"PyAEDT wave_port() returned None for {boundary_name!r}.")

    # PyAEDT 1.3's Driven-Terminal reference-conductor path ignores the public
    # ``modes=`` argument. HFSS requires one terminal mode per terminal, so bind
    # NumModes explicitly after creation (three at the coupled seam, one outside).
    boundary.props["NumModes"] = expected_terminal_count
    if not boundary.update():
        raise BuildError(f"Could not set {boundary_name!r} to {expected_terminal_count} mode(s).")

    post = list(app.oboundary.GetExcitationsOfType("Terminal"))
    new_terms = [term for term in post if term not in prior_terminals]
    if len(new_terms) != expected_terminal_count:
        raise BuildError(
            f"Physical port {physical_port!r} expected {expected_terminal_count} terminal(s), "
            f"created {len(new_terms)}."
        )

    for generated_name, semantic_name in zip(new_terms, terminal_names, strict=True):
        app.oboundary.RenameBoundary(generated_name, semantic_name)

    return TerminalRecord(
        physical_port=physical_port,
        boundary_name=boundary_name,
        terminal_names=terminal_names,
        terminal_objects=terminal_object_map,
        reference_conductors=list(ground_references),
        face_id=face_id,
        expected_terminal_count=expected_terminal_count,
        mode_count=int(boundary.props["NumModes"]),
    )


def _assign_ports(
    app: Hfss,
    transition_kind: str,
    assignments: list[PortRecord],
    ground_references: list[str],
    mtl_deembed_distance: float | str,
    single_trace_deembed_distance: float | str,
) -> list[TerminalRecord]:
    expected = _side_map(transition_kind)
    region_faces = {name: _find_region_face_on_side(app, expected[name]) for name in expected}

    if USE_SEAM_CENTER_GROUND_TERMINAL:
        if len(ground_references) < 3:
            raise BuildError(
                "Center-ground seam topology requires at least three finite-ground objects. "
                f"Got {len(ground_references)}: {ground_references!r}."
            )
        center_ground = _select_central_seam_ground(app, ground_references, assignments)
        seam_mode_names = ["o1", "o2", "g_center"]
        seam_object_map = {
            "o1": _object_map()["o1"],
            "o2": _object_map()["o2"],
            "g_center": center_ground,
        }
        seam_ref = [ground for ground in ground_references if ground != center_ground]
    else:
        center_ground = None
        seam_mode_names = CANONICAL_TERMINAL_ORDER[:2]
        seam_object_map = {"o1": _object_map()["o1"], "o2": _object_map()["o2"]}
        seam_ref = list(ground_references)

    outer_records = [r for r in assignments if r.physical_port in {"o3", "o4"}]

    prior = set(app.oboundary.GetExcitationsOfType("Terminal"))
    terminal_records: list[TerminalRecord] = []

    seam_region_face = region_faces["o1"]
    terminal_records.append(
        _create_wave_port(
            app,
            ground_references=seam_ref,
            terminal_names=seam_mode_names,
            terminal_object_map=seam_object_map,
            physical_port="seam",
            face_id=seam_region_face,
            boundary_name="seam",
            deembed_distance=mtl_deembed_distance,
            prior_terminals=prior,
        )
    )

    for rec in outer_records:
        face_id = region_faces[rec.physical_port]
        prior = set(app.oboundary.GetExcitationsOfType("Terminal"))
        terminal_records.append(
            _create_wave_port(
                app,
                ground_references=ground_references,
                physical_port=rec.physical_port,
                face_id=face_id,
                terminal_names=[rec.physical_port],
                terminal_object_map={rec.physical_port: _object_map()[rec.physical_port]},
                boundary_name=f"{rec.physical_port}_wave_port",
                deembed_distance=single_trace_deembed_distance,
                prior_terminals=prior,
            )
        )

    all_terminals = list(app.oboundary.GetExcitationsOfType("Terminal"))
    if len(all_terminals) != len(CANONICAL_TERMINAL_ORDER):
        raise BuildError(
            "Expected"
            f" {len(CANONICAL_TERMINAL_ORDER)} terminal excitations after assignment, "
            f"got {len(all_terminals)}."
        )

    if USE_SEAM_CENTER_GROUND_TERMINAL and center_ground is None:
        raise BuildError("Center-ground terminal mode enabled but no center ground was identified.")

    return terminal_records

### De-embed control readback

In [ ]:
TRANSITION_KIND = _normalize_transition_kind(TRANSITION_KIND)
MTL_DEEMBED_MM_EXPLICIT = _deembed_mm(MTL_DEEMBED_LENGTH_UM)
SINGLE_TRACE_DEEMBED_MM_EXPLICIT = _deembed_mm(SINGLE_TRACE_DEEMBED_LENGTH_UM)
MTL_DEEMBED_UM_EXPLICIT = _deembed_um(MTL_DEEMBED_LENGTH_UM)
SINGLE_TRACE_DEEMBED_UM_EXPLICIT = _deembed_um(SINGLE_TRACE_DEEMBED_LENGTH_UM)

print(f"Transition kind: {TRANSITION_KIND}")
print(
    "DEEMBED (explicit UI values): "
    f"MTL={MTL_DEEMBED_MM_EXPLICIT:.6f} mm, "
    f"SINGLE_TRACE={SINGLE_TRACE_DEEMBED_MM_EXPLICIT:.6f} mm"
)
print(
    "DEEMBED for PyAEDT wave_port: "
    f"MTL={MTL_DEEMBED_UM_EXPLICIT}, SINGLE_TRACE={SINGLE_TRACE_DEEMBED_UM_EXPLICIT}"
)
print(
    "Do not copy any guard-length derived convention into defaults: set "
    "MTL_DEEMBED_LENGTH_UM and SINGLE_TRACE_DEEMBED_LENGTH_UM explicitly when sweeping."
)
print(
    f"PyAEDT provenance: expected={PYAEDT_EXPECTED_VERSION}, "
    f"installed={PYAEDT_INSTALLED_VERSION or 'not-installed'}; "
    f"{PYAEDT_DOCS_URL} ; {PYAEDT_SOURCE_URL}"
)

## Create Simulation Component / Coupon

In [ ]:
transition, ordered_ports = _build_transition_component(
    TRANSITION_KIND,
    mtl_lead_length_um=MTL_LEAD_LENGTH_UM,
    single_trace_lead_length_um=SINGLE_TRACE_LEAD_LENGTH_UM,
)
coupon, port_records = _build_hfss_coupon_from_transition(
    transition,
    transition_kind=TRANSITION_KIND,
    mtl_lead_length_um=MTL_LEAD_LENGTH_UM,
    single_trace_lead_length_um=SINGLE_TRACE_LEAD_LENGTH_UM,
    mtl_deembed_length_um=MTL_DEEMBED_LENGTH_UM,
    single_trace_deembed_length_um=SINGLE_TRACE_DEEMBED_LENGTH_UM,
)

if coupon.info["hfss_coupon"]["transition_kind"] != TRANSITION_KIND:
    raise BuildError("Coupon metadata transition kind mismatch.")

display(coupon.plot())

df_ports = pd.DataFrame(
    [
        {
            "port": rec.physical_port,
            "object": rec.object_name,
            "side": rec.side,
            "axis": rec.axis,
            "side_sign": rec.side_sign,
            "port_center_um": rec.port_center_um,
        }
        for rec in port_records
    ]
)

display(df_ports)

print(f"Public ordered ports from cell: {ordered_ports}")

## Initialize AEDT Project / App

In [ ]:
if RUN_AEDT:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    hfss = Hfss(
        project=str(PROJECT_PATH),
        design=CASE_NAME,
        solution_type=SOLUTION_TYPE,
        version=AEDT_VERSION,
        non_graphical=NON_GRAPHICAL,
        new_desktop=True,
        close_on_exit=False,
    )
    hfss.modeler.model_units = "um"
    hfss.modeler.refresh_all_ids()
    if hfss.desktop_class.aedt_version_id != AEDT_VERSION:
        raise BuildError(
            f"AEDT version mismatch: requested {AEDT_VERSION!r}, "
            f"connected {hfss.desktop_class.aedt_version_id!r}."
        )
    build_model = bool(not hfss.modeler.object_names)
    print("Resuming existing project." if not build_model else "Created new HFSS project.")
else:
    hfss = None
    build_model = False
    print("AEDT is not started. Set RUN_PREPARE or RUN_SOLVER to initialize HFSS app.")

## Import GDS and Build the HFSS/Q3D/Q2D Model

In [ ]:
if RUN_AEDT and build_model:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    coupon.write_gds(GDS_PATH, with_metadata=False)

    layer_mapping = {
        HFSS_SIGNAL_P_LAYER[0]: (0.0, 0.0),
        HFSS_SIGNAL_R_LAYER[0]: (0.0, 0.0),
        HFSS_GROUND_LAYER[0]: (0.0, 0.0),
        HFSS_SUBSTRATE_LAYER[0]: (-SUBSTRATE_THICKNESS_UM, SUBSTRATE_THICKNESS_UM),
    }
    if ADD_GROUND_AIRBRIDGE:
        layer_mapping.update(
            {
                HFSS_AIRBRIDGE_VIA_LAYER[0]: (0.0, AIRBRIDGE_VIA_THICKNESS_UM),
                HFSS_AIRBRIDGE_LAYER[0]: (
                    AIRBRIDGE_VIA_THICKNESS_UM,
                    AIRBRIDGE_THICKNESS_UM,
                ),
            }
        )
    if not hfss.import_gds_3d(str(GDS_PATH), layer_mapping, units="um", import_method=1):
        raise BuildError(f"Could not import GDS: {GDS_PATH}")
    hfss.modeler.refresh_all_ids()

    airbridge_objects = []
    if ADD_GROUND_AIRBRIDGE:
        for layer_number, prefix in (
            (HFSS_AIRBRIDGE_LAYER[0], "airbridge_deck"),
            (HFSS_AIRBRIDGE_VIA_LAYER[0], "airbridge_via"),
        ):
            matches = [
                name
                for name in hfss.modeler.object_names
                if name.startswith(f"signal{layer_number}_")
            ]
            for index, selected in enumerate(matches, start=1):
                imported_obj = hfss.modeler.get_object_from_name(selected)
                imported_obj.name = f"{prefix}_{index}"
                airbridge_objects.append(imported_obj.name)

        hfss.modeler.refresh_all_ids()

    expected = (
        (HFSS_SIGNAL_P_LAYER[0], "signal_1"),
        (HFSS_SIGNAL_R_LAYER[0], "signal_2"),
        (HFSS_GROUND_LAYER[0], "finite_ground"),
        (HFSS_SUBSTRATE_LAYER[0], "substrate"),
    )
    imported = {}
    ground_references = []
    for layer_number, expected_name in expected:
        matches = [
            name for name in hfss.modeler.object_names if name.startswith(f"signal{layer_number}_")
        ]
        if not matches:
            raise BuildError(f"Missing imported object for {expected_name!r}.")
        if expected_name == "finite_ground":
            ground_references = []
            for index, selected in enumerate(matches, start=1):
                imported_obj = hfss.modeler.get_object_from_name(selected)
                imported_obj.name = f"finite_ground_{index}"
                ground_references.append(imported_obj.name)
            imported[expected_name] = ground_references
            continue
        if len(matches) != 1:
            raise BuildError(
                f"Expected one imported object for {expected_name!r}, got {matches!r}."
            )
        selected = matches[0]
        imported_obj = hfss.modeler.get_object_from_name(selected)
        imported_obj.name = expected_name
        imported[expected_name] = imported_obj.name

    hfss.modeler.refresh_all_ids()

## Geometry Verification

In [ ]:
if RUN_AEDT and build_model:
    display(
        pd.DataFrame({"role": list(imported), "object": [str(imported[role]) for role in imported]})
    )

## Materials and Boundaries

In [ ]:
if RUN_AEDT and build_model:
    hfss.modeler.get_object_from_name(
        "substrate"
    ).material_name = aedt_material_name_from_physical_key(SUBSTRATE_KEY)
    region = hfss.modeler.create_region(
        pad_value=_region_padding_for_kind(TRANSITION_KIND),
        pad_type="Absolute Offset",
        name="Region",
    )
    region.material_name = "vacuum"
    hfss.assign_perfect_e(
        ["signal_1", "signal_2", *ground_references, *airbridge_objects],
        name="PerfectE",
    )

## Ports / Nets / Excitations

In [ ]:
if RUN_AEDT and build_model:
    terminal_records = _assign_ports(
        hfss,
        transition_kind=TRANSITION_KIND,
        assignments=port_records,
        ground_references=ground_references,
        mtl_deembed_distance=MTL_DEEMBED_UM_EXPLICIT,
        single_trace_deembed_distance=SINGLE_TRACE_DEEMBED_UM_EXPLICIT,
    )

    all_terminals = list(hfss.oboundary.GetExcitationsOfType("Terminal"))
    if len(all_terminals) != len(CANONICAL_TERMINAL_ORDER):
        raise BuildError(
            f"Terminal inventory failed: expected {len(CANONICAL_TERMINAL_ORDER)}, "
            f"got {len(all_terminals)} -> {all_terminals!r}."
        )

    seam_record = next(
        (record for record in terminal_records if record.physical_port == "seam"),
        None,
    )
    seam_reference_grounds = list(seam_record.reference_conductors) if seam_record else []
    seam_subports = ["o1", "o2", "g_center"] if USE_SEAM_CENTER_GROUND_TERMINAL else ["o1", "o2"]

    metadata = {
        "transition_kind": TRANSITION_KIND,
        "mtl_lead_length_um": float(MTL_LEAD_LENGTH_UM),
        "single_trace_lead_length_um": float(SINGLE_TRACE_LEAD_LENGTH_UM),
        "mtl_deembed_length_um": float(MTL_DEEMBED_LENGTH_UM),
        "single_trace_deembed_length_um": float(SINGLE_TRACE_DEEMBED_LENGTH_UM),
        "deembed_mm_explicit": {
            "mtl": MTL_DEEMBED_MM_EXPLICIT,
            "single_trace": SINGLE_TRACE_DEEMBED_MM_EXPLICIT,
        },
        "finite_ground_width_um": float(FINITE_GROUND_WIDTH_UM),
        "mtl_deembed_um_explicit": MTL_DEEMBED_UM_EXPLICIT,
        "single_trace_deembed_um_explicit": SINGLE_TRACE_DEEMBED_UM_EXPLICIT,
        "frequency_ghz": [FREQUENCY_START_GHZ, FREQUENCY_STOP_GHZ, FREQUENCY_POINT_COUNT],
        "frequency_sweep_type": SWEEP_TYPE,
        "sweep_name": SWEEP_NAME,
        "airbridge": coupon.info["hfss_coupon"]["airbridge"],
        "airbridge_objects": airbridge_objects,
        "expected_terminal_order": CANONICAL_TERMINAL_ORDER,
        "use_seam_center_terminal": USE_SEAM_CENTER_GROUND_TERMINAL,
        "terminal_count": len(CANONICAL_TERMINAL_ORDER),
        "seam_reference_grounds": seam_reference_grounds,
        "pyaedt": {
            "expected_version": PYAEDT_EXPECTED_VERSION,
            "installed_version": PYAEDT_INSTALLED_VERSION or "not-installed",
            "docs_url": PYAEDT_DOCS_URL,
            "source_tags_url": PYAEDT_SOURCE_URL,
        },
        "physical_port_faces": {
            rec.physical_port: {
                "side": (
                    _side_map(TRANSITION_KIND)["o1"]
                    if rec.physical_port == "seam"
                    else _side_map(TRANSITION_KIND)[rec.physical_port]
                ),
                "face_id": rec.face_id,
                "expected_terminal_count": rec.expected_terminal_count,
                "mode_count": rec.mode_count,
            }
            for rec in terminal_records
        },
        "physical_port_faces_with_actual_order": [
            {
                "physical_port": rec.physical_port,
                "boundary_name": rec.boundary_name,
                "face_id": rec.face_id,
                "reference_conductors": rec.reference_conductors,
                "expected_terminal_count": rec.expected_terminal_count,
                "mode_count": rec.mode_count,
                "terminal_names": rec.terminal_names,
                "terminal_objects": rec.terminal_objects,
            }
            for rec in terminal_records
        ],
        "seam_subports": seam_subports,
        "region_padding_um": _region_padding_for_kind(TRANSITION_KIND),
        "solver_setup": {
            "name": SOLVER_SETUP_NAME,
            "solution_type": SOLUTION_TYPE,
            "max_adaptive_passes": MAX_ADAPTIVE_PASSES,
            "minimum_converged_passes": MINIMUM_CONVERGED_PASSES,
            "max_delta_s": MAX_DELTA_S,
            "maximum_delta_zo_percent": MAXIMUM_DELTA_ZO_PERCENT,
            "cpw_conductor_length_mesh_um": CPW_CONDUCTOR_LENGTH_MESH_UM,
            "cpw_conductor_length_mesh_max_additional_elements": (
                CPW_CONDUCTOR_LENGTH_MESH_MAX_ADDITIONAL_ELEMENTS
            ),
            "ground_surface_mesh_level": GROUND_SURFACE_MESH_LEVEL,
        },
    }

    RUN_DIR.mkdir(parents=True, exist_ok=True)
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"Saved review metadata: {METADATA_PATH}")

### Port assignment readback

In [ ]:
if RUN_AEDT and build_model:
    if not METADATA_PATH.exists():
        raise BuildError("Expected metadata file was not emitted.")

    metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    excitations = hfss.oboundary.GetExcitationsOfType("Terminal")
    if len(excitations) != len(CANONICAL_TERMINAL_ORDER):
        raise BuildError(
            f"Terminal validation failed: expected {len(CANONICAL_TERMINAL_ORDER)}, "
            f"got {len(excitations)} ({excitations!r})."
        )

    if metadata.get("expected_terminal_order") != list(CANONICAL_TERMINAL_ORDER):
        raise BuildError("expected_terminal_order metadata mismatch.")

    expected_face_sides = _side_map(TRANSITION_KIND)
    for rec in port_records:
        side = expected_face_sides[rec.physical_port]
        if rec.side != side:
            raise BuildError(f"Port {rec.physical_port!r} side mismatch: {rec.side!r} vs {side!r}.")

    print("Geometry and metadata validation passed.")

## Simulation Setup

In [ ]:
if RUN_AEDT and build_model:
    # This is a uniform CPW/MTL coupon, so signal and finite-ground conductors
    # share the W7 length mesh.  General HFSS ground planes use only the level-9
    # surface approximation unless a CPW/MTL study explicitly needs this rule.
    conductor_objects = ["signal_1", "signal_2", *ground_references]
    length_mesh = hfss.mesh.assign_length_mesh(
        assignment=conductor_objects,
        inside_selection=False,
        maximum_length=f"{CPW_CONDUCTOR_LENGTH_MESH_UM:g}um",
        maximum_elements=CPW_CONDUCTOR_LENGTH_MESH_MAX_ADDITIONAL_ELEMENTS,
        name="cpw_mtl_conductor_length_mesh",
    )
    ground_surface_mesh = hfss.mesh.assign_surface_mesh(
        assignment=ground_references,
        level=GROUND_SURFACE_MESH_LEVEL,
        name="ground_surface_fine_large",
    )
    if not length_mesh or not ground_surface_mesh:
        raise BuildError("HFSS mesh-operation assignment failed.")

    setup = (
        hfss.get_setup(SOLVER_SETUP_NAME)
        if SOLVER_SETUP_NAME in hfss.setup_names
        else hfss.create_setup(SOLVER_SETUP_NAME)
    )
    if not setup.enable_adaptive_setup_broadband(
        f"{FREQUENCY_START_GHZ}GHz",
        f"{FREQUENCY_STOP_GHZ}GHz",
        max_passes=MAX_ADAPTIVE_PASSES,
        max_delta_s=MAX_DELTA_S,
    ):
        raise BuildError("HFSS broadband setup failed.")

    # This extraction needs terminal S only. Saving every field made earlier
    # coupons unnecessarily large; turn fields on only for a dedicated field run.
    setup.props["SaveAnyFields"] = False
    setup.props["SaveRadFieldsOnly"] = False
    setup.props["MinimumConvergedPasses"] = MINIMUM_CONVERGED_PASSES
    # HFSS calls this GUI field "Maximum Delta Zo". It controls the Wave Port
    # 2D adaptive eigensolver, not the 3D Delta-S convergence or Lead stability.
    setup.props["PortAccuracy"] = MAXIMUM_DELTA_ZO_PERCENT
    if not setup.update():
        raise BuildError("Failed to update the HFSS adaptive and port-solver controls.")

## Simulation Configuration

In [ ]:
if RUN_AEDT:
    setup = hfss.get_setup(SOLVER_SETUP_NAME)
    print(
        "Wave Port Maximum Delta Zo readback: "
        f"{setup.props['PortAccuracy']}% (PyAEDT property: PortAccuracy)"
    )
    sweep = setup.get_sweep(SWEEP_NAME)
    if sweep:
        sweep.props.update(
            {
                "RangeType": "LinearCount",
                "RangeStart": f"{FREQUENCY_START_GHZ}GHz",
                "RangeEnd": f"{FREQUENCY_STOP_GHZ}GHz",
                "RangeCount": FREQUENCY_POINT_COUNT,
                "Type": SWEEP_TYPE,
                "SaveFields": False,
                "SaveRadFields": False,
            }
        )
        sweep.update()
    else:
        sweep = hfss.create_linear_count_sweep(
            SOLVER_SETUP_NAME,
            "GHz",
            FREQUENCY_START_GHZ,
            FREQUENCY_STOP_GHZ,
            num_of_freq_points=FREQUENCY_POINT_COUNT,
            name=SWEEP_NAME,
            save_fields=False,
            sweep_type=SWEEP_TYPE,
        )
        if not sweep:
            raise BuildError(f"Failed to create {SWEEP_TYPE} frequency sweep.")

    sweep_readback = {
        "type": str(sweep.props["Type"]),
        "start": str(sweep.props["RangeStart"]),
        "stop": str(sweep.props["RangeEnd"]),
        "point_count": int(sweep.props["RangeCount"]),
    }
    display(pd.DataFrame([sweep_readback]))
    if sweep_readback != {
        "type": SWEEP_TYPE,
        "start": f"{FREQUENCY_START_GHZ}GHz",
        "stop": f"{FREQUENCY_STOP_GHZ}GHz",
        "point_count": FREQUENCY_POINT_COUNT,
    }:
        raise BuildError(f"HFSS sweep readback mismatch: {sweep_readback}")

    hfss.save_project()

## Solve and Export

In [ ]:
if RUN_SOLVER and RUN_AEDT:
    setup = hfss.get_setup(SOLVER_SETUP_NAME)
    # Lead length changes the geometry, so every run starts from a fresh mesh.
    hfss.oanalysis.RevertSetupToInitial(SOLVER_SETUP_NAME)
    started = perf_counter()
    solved = hfss.analyze_setup(
        name=None,
        acf_file=str(ACF_PATH),
        blocking=False,
    )
    if not solved:
        raise BuildError(f"HFSS setup {SOLVER_SETUP_NAME} did not start.")

    # A blocking Analyze call makes a Human-run notebook look frozen. Poll the
    # same adaptive-pass profile shown in AEDT while the native solve continues.
    # AEDT 2024.2 may report idle briefly while handing a Fast sweep to a
    # background worker. Wait for a running state, then three idle polls.
    seen_running = False
    idle_polls = 0
    completed_passes = 0
    while not seen_running or idle_polls < 3:
        sleep(5)
        running = bool(hfss.desktop_class.are_there_simulations_running)
        seen_running = seen_running or running
        idle_polls = 0 if running else idle_polls + 1
        profiles = setup.get_profile()
        completed_passes = (
            max((profile.num_adaptive_passes for profile in profiles.values()), default=0)
            if profiles
            else completed_passes
        )
        clear_output(wait=True)
        display(
            HTML(
                f"<b>HFSS is {'running' if running else 'finishing'}</b> &middot; "
                f"{completed_passes} adaptive passes completed &middot; "
                f"{perf_counter() - started:.1f} s elapsed"
            )
        )

    completed_passes = max(
        (profile.num_adaptive_passes for profile in setup.get_profile().values()),
        default=completed_passes,
    )
    solve_seconds = perf_counter() - started
    clear_output(wait=True)
    display(
        HTML(
            f"<b>HFSS solve complete</b> &middot; "
            f"{completed_passes} adaptive passes &middot; {solve_seconds:.1f} s"
        )
    )
    timing = {
        "analyze_setup_seconds": solve_seconds,
        "completed_adaptive_passes": completed_passes,
        "transition_kind": TRANSITION_KIND,
        "mtl_lead_length_um": MTL_LEAD_LENGTH_UM,
        "single_trace_lead_length_um": SINGLE_TRACE_LEAD_LENGTH_UM,
        "mtl_deembed_length_um": MTL_DEEMBED_LENGTH_UM,
        "single_trace_deembed_length_um": SINGLE_TRACE_DEEMBED_LENGTH_UM,
        "maximum_delta_zo_percent": MAXIMUM_DELTA_ZO_PERCENT,
        "pyaedt_expected_version": PYAEDT_EXPECTED_VERSION,
        "pyaedt_installed_version": PYAEDT_INSTALLED_VERSION,
        "pyaedt_doc_url": PYAEDT_DOCS_URL,
        "pyaedt_source_url": PYAEDT_SOURCE_URL,
    }

    touchstone = hfss.export_touchstone(
        setup=SOLVER_SETUP_NAME,
        sweep=SWEEP_NAME,
        output_file=str(TOUCHSTONE_PATH),
    )
    if not touchstone:
        raise BuildError(f"Touchstone export failed for {TOUCHSTONE_PATH}")

    touchstone_normalization = _normalize_touchstone_ports(
        TOUCHSTONE_PATH,
        CANONICAL_TERMINAL_ORDER,
        form="ma",
        r_ref=50.0,
    )

    # Port Zo comes from Modal Solution Data, not Terminal Touchstone metadata.
    # With no impedance line this is HFSS's Zpi definition and is used only to
    # diagnose whether the wave-port reference planes are far enough away.
    port_zo_expressions = [f"Zo({name})" for name in CANONICAL_TERMINAL_ORDER]
    port_zo = hfss.post.get_solution_data(
        expressions=port_zo_expressions,
        setup_sweep_name=f"{SOLVER_SETUP_NAME} : {SWEEP_NAME}",
        report_category="Modal Solution Data",
    )
    if not port_zo:
        raise BuildError("Modal Solution Data Port Zo extraction failed.")
    frequency_ghz, _ = port_zo.get_expression_data(port_zo_expressions[0], "real")
    port_zo_columns = {"frequency_ghz": frequency_ghz}
    for expression in port_zo_expressions:
        _, real = port_zo.get_expression_data(expression, "real")
        _, imag = port_zo.get_expression_data(expression, "imag")
        port_zo_columns[f"Re({expression})_ohm"] = real
        port_zo_columns[f"Im({expression})_ohm"] = imag
    pd.DataFrame(port_zo_columns).to_csv(PORT_ZO_PATH, index=False)

    if METADATA_PATH.exists():
        metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
        metadata["touchstone_path"] = str(TOUCHSTONE_PATH)
        metadata["touchstone_normalization"] = touchstone_normalization
        metadata["touchstone_form"] = "ma"
        metadata["touchstone_reference_ohm"] = 50.0
        metadata["modal_port_zo_zpi_path"] = str(PORT_ZO_PATH)
        metadata["quantity_authority"] = {
            "terminal_scattering": "Terminal Solution Data / St(...) / Terminal Touchstone",
            "port_zo_zpi": (
                "Modal Solution Data / Port Zo / "
                f"Zo({', '.join(CANONICAL_TERMINAL_ORDER)}); no impedance line"
            ),
        }
        METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    timing["touchstone_path"] = str(TOUCHSTONE_PATH)
    timing["modal_port_zo_zpi_path"] = str(PORT_ZO_PATH)
    timing.update(
        {f"touchstone_{name}": value for name, value in touchstone_normalization.items()},
    )
    TIMING_PATH.write_text(json.dumps(timing, indent=2), encoding="utf-8")

    print(f"Touchstone written to: {touchstone}")
    print(f"Timing written to: {TIMING_PATH}")
elif not RUN_SOLVER:
    print("No solve requested; set RUN_SOLVER=True when solver execution is desired.")

## Adaptive-Pass Convergence / Solver Diagnostics

In [ ]:
if TIMING_PATH.exists():
    print(TIMING_PATH.read_text(encoding="utf-8"))
else:
    print("No timing data yet; run with RUN_SOLVER=True.")

## Results: Plots and Readable Tables

### Physics Analysis Results

In [ ]:
if METADATA_PATH.exists():
    metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    preview_records = []
    for item in metadata.get("physical_port_faces_with_actual_order", []):
        preview_records.append(
            {
                "physical_port": item.get("physical_port"),
                "face_id": item.get("face_id"),
                "boundary": item.get("boundary_name"),
                "expected_terminal_count": item.get("expected_terminal_count"),
                "terminal_names": ",".join(item.get("terminal_names", [])),
                "terminal_objects": ",".join(
                    f"{name}:{object_name}"
                    for name, object_name in item.get("terminal_objects", {}).items()
                ),
            }
        )
    display(pd.DataFrame(preview_records))
    display(
        pd.DataFrame(
            [
                {
                    "run_id": "review",
                    "transition": metadata.get("transition_kind"),
                    "mtl_lead_length_um": metadata.get("mtl_lead_length_um"),
                    "single_trace_lead_length_um": metadata.get("single_trace_lead_length_um"),
                    "mtl_deembed_length_um": metadata.get("mtl_deembed_length_um"),
                    "single_trace_deembed_length_um": metadata.get(
                        "single_trace_deembed_length_um"
                    ),
                    "deembed_mm_explicit": metadata.get("deembed_mm_explicit"),
                    "frequency_GHz": "-".join(str(x) for x in metadata.get("frequency_ghz", [])),
                    "metadata_path": str(METADATA_PATH),
                }
            ]
        )
    )

### Simulation Performance / Benchmarks

In [ ]:
if TIMING_PATH.exists():
    display(pd.DataFrame([json.loads(TIMING_PATH.read_text(encoding="utf-8"))]))
else:
    print("No timing data available.")

## Save and Release AEDT

In [ ]:
if RUN_AEDT:
    hfss.save_project()
    if CLOSE_DESKTOP:
        hfss.release_desktop(close_projects=True, close_desktop=True)
        print("AEDT session closed.")
    else:
        print("AEDT project remains open for GUI inspection.")